# Segment Anything
Are you a purist at heart?

If so, you might want to create your own image segmentation model from scratch - have fun!

But let’s say you’re on the job, then you might be wasting your time reinventing the wheel.

In this notebook, I explore using the “FastSAM” model from Ultralytics.

# What is SAM (Segment Anything Model)
The Segment Anything Model (SAM) is a promptable segmentation model -- that is, the segmentation is suggestible. Give it an
image coordinate or text description and it will attempt to identify the object. 
SAM works right out of the box -- no fine-tuning necessary for many purposes. Importantly, it
is license under Apache 2.0, which is an extremely permissive license.

SAM was originally developed at Meta and open-sourced
on [GitHub](https://github.com/facebookresearch/segment-anything), which provides 
an aggressively concise description:

> The [Segment Anything Model (SAM)](https://arxiv.org/abs/2304.02643) produces high quality object masks from input prompts such as points or boxes, and it can be used to generate masks for all objects in an image. It has been trained on a dataset of 11 million images and 1.1 billion masks, and has strong zero-shot performance on a variety of segmentation tasks.

A more full-bodied description from the [Ultralytics website](https://docs.ultralytics.com/models/sam/):

> The Segment Anything Model, or SAM, is a cutting-edge image segmentation model that allows for promptable segmentation, providing unparalleled versatility in image analysis tasks. SAM forms the heart of the Segment Anything initiative, a groundbreaking project that introduces a novel model, task, and dataset for image segmentation.
> 
> SAM's advanced design allows it to adapt to new image distributions and tasks without prior knowledge, a feature known as zero-shot transfer. Trained on the expansive SA-1B dataset, which contains more than 1 billion masks spread over 11 million carefully curated images, SAM has displayed impressive zero-shot performance, surpassing previous fully supervised results in many cases.

The most robust explanation and details, without actually reading the [paper](https://arxiv.org/abs/2304.02643),
can be found on [Meta's AI blog](https://ai.meta.com/blog/segment-anything-foundation-model-image-segmentation/).

There is also a [website](https://segment-anything.com) dedicated to SAM, which has a nice, 
straight-to-the-point FAQs section.

Play around with the model using the online [demo](https://segment-anything.com/demo).

Learn more about it perusing some Jupyter notebooks:
* Facebook Research: [Automatically generating object masks with SAM](https://github.com/facebookresearch/segment-anything/blob/main/notebooks/automatic_mask_generator_example.ipynb)
* [Roboflow](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/how-to-segment-anything-with-sam.ipynb)

**NOTE**: SAM2 ([link1](https://ai.meta.com/sam2/), [link2](https://ai.meta.com/blog/segment-anything-2/))
is available now too; it works out-of-the-box on video.

# FastSAM
FastSAM is basically SAM with a small tradeoff between size and accuracy.

From [HuggingFace](https://huggingface.co):
> The Fast Segment Anything Model(FastSAM) is a CNN Segment Anything Model trained by only 2% of the SA-1B dataset published by SAM authors. The FastSAM achieve a comparable performance with the SAM method at 50× higher run-time speed.

The Chinese Academy of Sciences Image and Video Analysis (CASIA IVA) group is responsible for the development of [FastSAM](https://arxiv.org/abs/2306.12156) -- a fast, accurate implementation of SAM.  FastSAM provides high-performance image segmentation while optimizing for speed and ease of use.  One can `pip install` the FastSAM model directly from their [GitHub](https://github.com/CASIA-IVA-Lab/FastSAM). [HuggingFace](https://huggingface.co) provides an [overview](https://huggingface.co/An-619/FastSAM) on FastSAM using this repository. [Roboflow](https://roboflow.com) provides a much more in-depth [notebook tutorial](https://github.com/roboflow/notebooks/blob/main/notebooks/how-to-segment-anything-with-fast-sam.ipynb).

[Ultralytics](https://www.ultralytics.com/) is a company that specializes in developing cutting-edge, easy-to-use, and high-performance deep learning models, particularly for computer vision tasks. They are well-known for their contributions to the YOLO (You Only Look Once) series of models. They also provide access to FastSAM through their `ultralytics`
package (which I use below), and provide the following [tutorial](https://docs.ultralytics.com/models/fast-sam/).

**NOTE**: There is also a "faster SAM" called [MobileSAM](https://github.com/ChaoningZhang/MobileSAM). 


# Environment

This notebook uses two related, but not identical, FastSAM setups.

Most of the notebook uses the `ultralytics` package directly:

```python
from ultralytics import FastSAM
```

The later CASIA section uses a local clone of the CASIA FastSAM repository in `./FastSAM`. The helper script fetches that clone automatically and pins it to this commit:

```text
b4ed20c2fed75eadc5aa7d8b09fedd137b873b52
```

That commit matches the local copy that was downloaded on August 15, 2024, apart from one local patch described below.

```python
sys.path.insert(1, f'{os.getcwd()}/FastSAM')
from fastsam import FastSAM, FastSAMPrompt
```

To make this reproducible without polluting global Conda environments, use the local helper script in this folder.

First make it executable:

```bash
chmod +x fastsam-env.sh
```

For the main Ultralytics path:

```bash
./fastsam-env.sh ultralytics
```

For the later CASIA path:

```bash
./fastsam-env.sh casia
```

The script creates project-local Conda environments:

```text
.conda-fastsam-ultralytics/
.conda-fastsam-casia/
```

Those folders are intentionally ignored by `.gitignore`. The fetched `FastSAM/` clone is also ignored so third-party source code is not committed to this project. The `weights/` folder is ignored too because the checkpoint files are large local artifacts and exceed normal GitHub file-size limits.

## Required Local Files

The notebook expects these local assets:

```text
fastsam.ipynb
fastsam-env.sh
utils.py
utils_casia.py

images/
  kevin.jpg
  roboflow-dog.jpeg
  kitty.png

weights/  # local-only, ignored by git
  FastSAM-s.pt
  FastSAM-x.pt
  FastSAM.pt

outputs/
  *.png  # linked markdown images saved from previous notebook runs

FastSAM/  # fetched automatically by ./fastsam-env.sh casia; ignored by git
```

## What The Script Installs

The `ultralytics` environment installs:

```bash
python -m pip install torch torchvision
python -m pip install ultralytics==8.2.65
python -m pip install git+https://github.com/openai/CLIP.git
```

plus notebook/plotting basics such as `jupyter`, `ipykernel`, `pandas`, `seaborn`, `matplotlib`, `pillow`, and `numpy`.

The `casia` environment installs the same notebook/plotting basics, PyTorch, OpenAI CLIP, and the runtime packages needed by the local CASIA FastSAM clone, including `opencv-python`, `PyYAML`, `requests`, `scipy`, and `tqdm`.

I do not install `FastSAM/requirements.txt` verbatim in the helper script because it includes app/demo dependencies that are not needed by this notebook. If you want to run CASIA's Gradio app or original demo scripts, use their full requirements file separately.

## Notes

- Before running model cells, make sure the local `weights/` folder contains `FastSAM-s.pt`, `FastSAM-x.pt`, and `FastSAM.pt`. These are not committed.
- The notebook was originally run on a MacBook with Python 3.12.
- The helper script creates Python 3.12 environments.
- The `ultralytics` version is pinned because `FastSAMPrompt` behavior changed across nearby releases.
- The CASIA section depends on the local `FastSAM/` folder. The helper script checks out the pinned commit and reapplies the local `FastSAM/fastsam/prompt.py` change where `import clip` is added.

# Resize the Image to 1024x1024
This is necessary to use with the FastSAM model.

In [ ]:
from PIL import Image
raw_image = Image.open("images/kevin.jpg")

In [ ]:
def resize_image(image, input_size):
    w, h = image.size
    scale = input_size / max(w, h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    image = image.resize((new_w, new_h))
    return image

In [ ]:
import matplotlib.pyplot as plt
resized_image = resize_image(raw_image, input_size=1024)
plt.imshow(resized_image);

<!-- saved-output: cell 6 -->
![Resized selfie image](outputs/selfie-resized-1024.png)



# Using a CUDA GPU, MPS GPU, or CPU?
Here is some code to figure that out, and how to choose.

In [ ]:
import torch

def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("CUDA is available. Using GPU.")
    # Check for MPS (Apple Silicon)
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("MPS is available. Using Apple GPU.")
    # Default to CPU
    else:
        device = torch.device("cpu")
        print("Using CPU.")
    return device
#--
device = get_device()
device

# The FastSAM Model

This section expects the `ultralytics` environment:

```bash
./fastsam-env.sh ultralytics
```

The important package pin is:

```bash
python -m pip install ultralytics==8.2.65
```

With that environment active, the model can be loaded from a local checkpoint in `weights/`:

In [ ]:
from ultralytics import FastSAM
model = FastSAM("weights/FastSAM-s.pt")  # or another FastSAM checkpoint

# Detecting, bounding, and segmenting all the objects
The FastSAM model will essentially try to identify everything that appears to be 
a separate object. 

In [ ]:
results = model(resized_image, device=device, retina_masks=True,
                conf=0.6, iou=0.9)

The result is a single-element list that contains a list of `Results` objects. 

In [ ]:
results = results[0]

For now, let's just look at a single `Results` object that we will call `obj`.

In [ ]:
obj = results[1]

What do these objects contain?

Well, for one, each `Results` objects contains a normalized version of the original image.

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(obj.orig_img)

<!-- saved-output: cell 18 -->
![Original image stored on a FastSAM result object](outputs/selfie-result-original-image.png)



### Bounding Boxes
Each `Results` object contains a `Boxes` object.

In [ ]:
obj.boxes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
def bounding_box_overlay(resized_image, results_objects, linewidth=2, edgecolor='orange', 
                      alpha=1.0, first_box_color=None, first_box_line=None, ax=None):
    if first_box_color is None: first_box_color = edgecolor
    color_map = {0:first_box_color}
    if first_box_line is None: first_box_line = linewidth
    line_map = {0:first_box_line}
    if ax is None:
        fig, ax = plt.subplots(1)
    ax.imshow(resized_image)
    for idx,obj in enumerate(results_objects):
        color = color_map.get(idx,edgecolor)
        lwidth = line_map.get(idx,linewidth)
        x1, y1, x2, y2 = obj.summary()[0]['box'].values()
        w = x2-x1
        h = y2-y1
        # Create a Rectangle patch
        rect = patches.Rectangle((x1, y1), w, h, linewidth=lwidth, edgecolor=color, 
                                 facecolor='none', alpha=alpha)
        ax.add_patch(rect)

In [ ]:
bounding_box_overlay(resized_image, results, first_box_color='red')

<!-- saved-output: cell 22 -->
![Selfie bounding boxes sorted by confidence](outputs/selfie-boxes-by-confidence.png)



# Excursion: Overlay or Grid?
Overlays are nice, but sometimes a grid is better -- e.g., for showing which
bounding box comes first, second, and third.  

This is also true of plotting segmentation masks and segmentation contours, so
here we define a general grid plotting function that takes in an "overlay function"
as an argument. (This is a use of DRY.)

In [ ]:
def plot_grid(resized_image, results_objects, overlay_function, titles=None, n_cols=3, alpha=0.75, **kwargs):
    """
    Shows grid of result masks
    """
    n_images = len(results_objects)
    
    # Validation check: Ensure titles are provided and match the number of result objects
    if isinstance(titles,str): titles = [titles]
    if titles and len(titles) != n_images:
        raise ValueError("The number of titles must match the number of result objects.")

    n_rows = (n_images + n_cols - 1) // n_cols  # Calculate the number of rows needed
    
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
    axs = axs.flatten()  # Flatten in case of multi-row subplots
    
    for idx, obj in enumerate(results_objects):
        plt.sca(axs[idx])  # Set the current axes to the subplot
        overlay_function(resized_image, obj, alpha=alpha, ax=axs[idx], **kwargs)  # Use the provided function
        axs[idx].axis('off')  # Optional: turn off axis labels
        if titles:
            axs[idx].set_title(titles[idx], fontsize=10)
    
    # If there are more subplots than images, hide the unused subplots
    for idx in range(n_images, n_rows * n_cols):
        axs[idx].axis('off')

    plt.tight_layout()
    plt.show()

### Grid of Bounding Box Plots

In [ ]:
def bounding_box_grid(resized_image, results_objects, titles=None, n_cols=3, **kwargs):
    plot_grid(resized_image, results_objects, bounding_box_overlay, titles=titles, n_cols=n_cols, **kwargs)

In [ ]:
bounding_box_grid(resized_image, results[:3], edgecolor='red')

<!-- saved-output: cell 27 -->
![Top three selfie bounding boxes by confidence](outputs/selfie-top3-confidence-box-grid.png)



# Ok - How to get at an object you care more about?
Notice how the first bounding box is around a window? Do we care about that specific window? Do
we care about windows at all?

What if we want to automate a script that finds the most prominent object in the image? Or 
we want to take more control over what is considered the first bounding box in the list of
object detections?

Spoiler Alert: You can use FastSAM's text prompting ability. 

But let's forget that exists for a moment!  Let's use what we already have on hand... 

The results list above, by default, is organized by the confidence score of the bounding box. 

In [ ]:
conf = torch.tensor([],device=device)
for ooo in results: 
    conf = torch.cat((conf,ooo.boxes.conf))
conf[:10]

Are there other ways to order this list? 

Sure! For example, if we assume that the most important object in any image is the largest object,
then we can order the object detections list by area of the segmentation mask or bounding box.

In [ ]:
area_sm = torch.tensor([]) # Area of Segmentation Masks
area_bb = torch.tensor([]) # Area of Bounding Boxes
for object in results: 
    mask = object.masks.data == 1.0
    area_sm = torch.cat((area_sm, torch.tensor([mask.sum()])))
    box = object.boxes.xywh[0,2:]
    area_bb = torch.cat((area_bb, torch.tensor([box.prod()])))
#--
print("\nSegmentation Mask Areas: \n\t", area_sm[:10])
print("\nBounding Box Areas: \n\t", area_bb[:10])

We showed the mask and box areas associated with each `Results` object, but to actually
sort the results list by area we can use Python's `sorted` with a lambda function: 

In [ ]:
area_results = sorted(results, key=lambda x: x.masks.data.sum(), reverse=True)
area_results_bb = sorted(results, key=lambda x: x.boxes.xywh[0,2:].prod(), reverse=True)
#--
print("\nResults objects sorted by mask area: \n\t",[result.masks.data.cpu().numpy().sum().astype(int) for result in area_results[:10]])
print("\nResults objects sorted by box area: \n\t",[result.boxes.xywh[0,2:].cpu().numpy().prod().astype(int) for result in area_results_bb[:10]])


**And now draw the results when reverse-sorted by mask area -- top results is in red.**

In [ ]:
bounding_box_overlay(resized_image, area_results, first_box_color='red', first_box_line=5)

<!-- saved-output: cell 35 -->
![Selfie bounding boxes sorted by mask area](outputs/selfie-boxes-by-mask-area.png)



**The first 3 objects now correspond to the human, jacket, and face.**

In [ ]:
bounding_box_grid(resized_image, area_results[:3], edgecolor='red', titles=['Human','Jacket','Face'])

<!-- saved-output: cell 37 -->
![Top three selfie bounding boxes by mask area](outputs/selfie-top3-mask-area-box-grid.png)



It's a bounding box around me!

## Masks Objects
Each `Results` object also contains a `Masks` object, which contains the 
segmentation information and a mask array for that segmentation. We used this information
above to sort the results by segmentation mask area. 

The code below focuses a bit more on these `Mask` objects, and we create a way
to overlay the masks onto the original image.


Let's look at the **segmentation mask**.

In [ ]:
import numpy as np
def get_mask(obj):
    mask = obj.masks.data[0].to('cpu').numpy()
    return mask

def segmentation_mask_overlay(resized_image, results_objects, alpha=0.75, ax=None):
    """
    Shows result's mask if only one Results object,
    Else creates union of masks
    """
    if ax is None:
        fig, ax = plt.subplots(1)
    ax.imshow(resized_image)
    # Initialize mask
    mask = get_mask(results_objects[0])
    combined_mask = np.zeros_like(mask, dtype=bool)
    # Create mask
    for obj in results_objects:
        mask = get_mask(obj)
        combined_mask = np.logical_or(combined_mask, mask)  # Union of masks
    ax.imshow(combined_mask, alpha=alpha)

def segmentation_mask_grid(resized_image, results_objects, titles=None, n_cols=3, alpha=0.75, **kwargs):
    plot_grid(resized_image, results_objects, segmentation_mask_overlay, titles=titles, n_cols=n_cols, alpha=alpha, **kwargs)


#### Segmentation Mask Overlay
Sometimes you might want to combine masks. Here are the first 3 results by confidence level.

In [ ]:
segmentation_mask_overlay(resized_image, results[:3])

<!-- saved-output: cell 43 -->
![Combined mask overlay for top three confidence results](outputs/selfie-top3-confidence-mask-overlay.png)



#### Segmentation Mask Grid
Other times you might not want to combine masks. Here are the first 3 masks by bounding box area.

In [ ]:
segmentation_mask_grid(resized_image, area_results[:3], titles=['Human','Jacket','Face'])

<!-- saved-output: cell 45 -->
![Mask grid for top three mask-area results](outputs/selfie-top3-mask-area-mask-grid.png)



### Segmentation Contours

In [ ]:
def segmentation_contour_overlay(resized_image, results_objects, 
        first_obj_edge='red', first_obj_face='orange', first_obj_line=3, alpha=0.5, ax=None):
    edge_map = {0:first_obj_edge}
    face_map = {0:first_obj_face}
    line_map = {0:first_obj_line}
    if ax is None:
        fig, ax = plt.subplots(1)
    ax.imshow(resized_image)
    for idx,obj in enumerate(results_objects):
        edgecolor = edge_map.get(idx,'blue')
        facecolor = face_map.get(idx,'yellow')
        linewidth = line_map.get(idx,2)
        xy = obj.masks.xy[0]
        # Create a Polygon patch
        polygon = patches.Polygon(xy, closed=True, linewidth=linewidth, 
                        edgecolor=edgecolor, facecolor=facecolor, alpha=alpha)
        ax.add_patch(polygon)

def segmentation_contour_grid(resized_image, results_objects, titles=None, n_cols=3, alpha=0.5, **kwargs):
    plot_grid(resized_image, results_objects, segmentation_contour_overlay, 
              titles=titles, n_cols=n_cols, alpha=alpha, **kwargs)

Here, we make an interesting use of both the overlay and grid functions.

Instead of passing a Results object containing one or more Results objects, we will
pass a list of overlays we want to see on the grid.

In [ ]:
human = area_results[0]
human_and_jacket = area_results[:2]
jacket_and_face = area_results[1:3]
overlays = [human, human_and_jacket, jacket_and_face]

segmentation_contour_grid(resized_image, overlays, titles=['Human', 'Human and Jacket', 'Jacket and Face'])

<!-- saved-output: cell 49 -->
![Contour overlays for selected selfie result combinations](outputs/selfie-contour-overlays.png)



# FastSAM Prompting
In the case of my selfie, sorting by area let us gain access to `Results` object
that is of most interest in the pic, but this is somewhat based on good luck: 
the object(s) one cares about in an image will not always be identified by having 
the largest area(s)!

**Image Coordinates**: One way to solve this is by using image coordinates, then identifying which masks contain those image coordinates. However, relying on image coordinates means that we need to know the image coordinates! In an application setting,
one way to deal with this is adding functionality that allows the user to click on the object(s) they care about. The `ultralytics`
package can then be used to do the rest via point-prompt processing feature.

**Text Prompts**: For a more programmatic automated approach, we might want something else. Text prompting would be nice!
And the `ultralytics` package has this functionality. Text prompting is still compatible with user interaction if
desired, but can also be used more easily in routine tasks, like an app that focuses exclusively on 
cat, dog, and bird identification.

In version `8.2.65`, the prompt processing features are available via the `FastSAMPrompt`
object, which takes a list of `Results` objects upon instantiation and then provides
methods for searching those results.  The `text_prompt` method
finds the objects most closely matching the text prompt (e.g., "cat or dog or bird").


# Failed Attempt \#1: Are close-Up selfies bad for FastSAM?
When I began tinkering with the `ultralytics` package, it was with a selfie. Little 
did I know that FastSAM simply could not semantically understand such a close-up picture
of a face. Or perhaps just something is just tricky about this particular selfie. Either way:
text prompting was not working, and it was bumming me out.

I tried prompting with "person", with "face", with "tree", and even with "skibbidy dee." Each time, the
model would return the same results -- and it was never a person, face, tree, or skibbidy dee! 
 
### Debugging

1. Figured I was doing something wrong, but I wasn't: my code was exactly as written in the ultralytics docs.
2. Blamed it on the conda installation. Made a new conda environment entirely with pip installs. Nope! Same bogus results.
3. Assumed the result must be due to some mystery interaction (arising in both environments) between the ultralytics package and the fact I was running things on my Macbook (e.g., it's not Linux, it's not using CUDA GPUs). Could have tried using a Docker environment at this point, but instead created third conda PyTorch environment where I installed the `fastsam` package from GitHub, like in the [RoboFlow tutorial](https://github.com/roboflow/notebooks/blob/main/notebooks/how-to-segment-anything-with-fast-sam.ipynb):
  * `git clone https://github.com/CASIA-IVA-Lab/FastSAM.git`
  * `pip -q install -r FastSAM/requirements.txt`
  * `pip -q install git+https://github.com/openai/CLIP.git`

Nothing worked! What was going on?

It wasn't the code I was using. It wasn't the environment. 

**The bug was the image itself!**

I found this out by recreating the environment from the [RoboFlow notebook](https://github.com/roboflow/notebooks/blob/main/notebooks/how-to-segment-anything-with-fast-sam.ipynb) and running it on my Macbook. 

Their text prompt example worked just fine! **"Ok,"** I thought, **"Let's run this code with my selfie."**
This time it didn't work! Same problems. So then I thought, **"Ok, let me try their image example
with my code in my other conda environments."** 

It worked, but not in the `ultralytics` environment I set up. It only worked in the CASIA environment
I set up, similar to the RoboFlow environment.

**The bug is also the ultralytics conda environment.**

------

As an aside, it also became apparent how the performance of the different versions
of FastSAM compared: `FastSAM.pt` works much better than `FastSAM-s.pt` (go figure, it's
a much bigger model!).

------

# Failed Attempt \#2: `FastSAMPrompt` Crashes a Few Days Later
Can't emphasize enough how important it is to specify version numbers
on certain packages when creating environments -- in this case the `ultralytics` package. For using text
prompts in the environment I created on July 25, `from ultralytics.models.fastsam import FastSAMPrompt`
worked in the version installed by `pip install ultralytics` (version `8.2.65`). The
version installed 3 days later (`8.2.68`) when I recreated the environment no longer 
supported this syntax.

In version `8.2.65` (July 25), one can use this code to textually prompt the model:

```python
from ultralytics import FastSAM
from ultralytics.models.fastsam import FastSAMPrompt
model = FastSAM("FastSAM-s.pt")  # or FastSAM-x.pt
results = model(resized_image, device=device, retina_masks=True,
                conf=0.6, iou=0.9)[0]
prompt = FastSAMPrompt(resized_image, results[0], device=device)
prompt_result = prompt.text_prompt(text='person')
draw_bounding_box(resized_image, prompt_result[0])
```

In version `8.2.77` (Aug 13), that code doesn't work! But this much simpler
snippet appears to do the same thing:

```python
from ultralytics import FastSAM
model = FastSAM("FastSAM-s.pt")
results = model(resized_image, texts="person")
draw_bounding_box(resized_image, results[0])
```

----

**Question**: Does any of the updated `ultralytics` packages (`8.2.68` through `8.2.77`) 
come with the power to text prompt my up-close selfie with "person" or "face" using FastSAM? 

**Answer**: Nope!

----

For consistency with the most of this notebook/blog, I use `ultralytics` version 
`8.2.65` below to show these failed attempts.

For sanity's sake, I put a bunch of the code and functions above into a utils package, 
which is used below.

# Explicit Failure Details!

In [ ]:
from utils import get_device, get_results, bounding_box_overlay
from ultralytics.models.fastsam import FastSAMPrompt

device = get_device()

fastsam_s = 'weights/FastSAM-s.pt' # downloads automatically if not there
fastsam_x = 'weights/FastSAM-x.pt' # downloads automatically if not there
fastsam_hf= 'weights/FastSAM.pt'

### Failure on Selfie

In [ ]:
# Small FastSAM and Big FastSAM Results
selfie_results_fs = get_results(resized_image, fastsam_s, device=device)
selfie_results_fx = get_results(resized_image, fastsam_x, device=device)

# Prompters
selfie_prompter_fs = FastSAMPrompt(resized_image, selfie_results_fs, device=device)
selfie_prompter_fx = FastSAMPrompt(resized_image, selfie_results_fx, device=device)

# Prompts
selfie_prompt_person_fs = selfie_prompter_fs.text_prompt(text='person')
selfie_prompt_person_fx = selfie_prompter_fs.text_prompt(text='person')
selfie_prompt_face_fx   = selfie_prompter_fs.text_prompt(text='face')
selfie_prompt_tree_fx   = selfie_prompter_fs.text_prompt(text='tree')

# Grid of Fs-vs-Fx Overlays
overlays = [selfie_prompt_person_fs, selfie_prompt_person_fx]
titles = ['Person (FastSAM-s)', 'Person (FastSAM-x)']
bounding_box_grid(resized_image, overlays, titles=titles, first_box_color='red')

# Grid of Other Fx Overlays
overlays = [selfie_prompt_face_fx, selfie_prompt_tree_fx]
titles = ['Face (FastSAM-x)', 'Tree (FastSAM-x)']
bounding_box_grid(resized_image, overlays, titles=titles, first_box_color='red')

<!-- saved-output: cell 53 -->
![Selfie text prompt for person using FastSAM-s and FastSAM-x](outputs/selfie-prompt-person-fastsam-s-vs-x.png)

![Selfie text prompts for face and tree](outputs/selfie-prompt-face-tree-fastsam-x.png)



### Failure on Roboflow Dog
Below I use very similar code to above, but modified to analyze the Roboflow dog. Also, instead
of explicitly showing all the code again, I call a wrapper function I wrote from my `utils` library.

In [ ]:
from utils import get_raw_image, get_prompt_results

img = get_raw_image('images/roboflow-dog.jpeg')
prompts = ['cap','dog','bag','building']
# prompt_results_fs = get_prompt_results(img, 'weights/FastSAM-s.pt', prompts)
# prompt_results_fx = get_prompt_results(img, 'weights/FastSAM-x.pt', prompts)
prompt_results_hf = get_prompt_results(img, fastsam_hf, prompts) # Weights from RoboFlow notebook / HuggingFace

# Grid of Fs Overlays
overlays = prompt_results_hf
titles = ['Cap (FastSAM-s)', 'Dog (FastSAM-s)',  'Bag (FastSAM-s)',  'Building (FastSAM-s)']
bounding_box_grid(img, overlays, titles=titles, n_cols=4, first_box_color='red')

<!-- saved-output: cell 55 -->
![Roboflow dog text prompts using the Hugging Face FastSAM weights](outputs/roboflow-dog-text-prompts-hf.png)



I tinkered with this for a while but didn't get it to work. I did get some things to work in the CASIA environment
below, but at the expense of not being able to use the overlay and grid plotting functions I designed above.

# CASIA Environment

This section expects the separate CASIA environment:

```bash
./fastsam-env.sh casia
```

It also expects the local CASIA clone at `./FastSAM`. `./fastsam-env.sh casia` fetches that clone automatically, checks out the pinned commit, and reapplies the local `import clip` patch. The code below temporarily adds that folder to `sys.path` so this import resolves to the local clone rather than the `ultralytics` package:

```python
from fastsam import FastSAM, FastSAMPrompt
```

**NOTE**: The original local copy had exactly one source change relative to the pinned FastSAM commit: `FastSAM/fastsam/prompt.py` needed `import clip`. The environment helper now applies that patch automatically.

Other practical differences from the Ultralytics section:

- Prompt outputs are NumPy arrays instead of `Results` objects.
- The overlay and grid plotting helpers written for Ultralytics `Results` objects do not work directly here.
- This section is mostly useful for reproducing the CASIA/Roboflow-style text-prompt behavior.

In [ ]:
# Had to modify `prompt.py` file -- "import clip"
import matplotlib.pyplot as plt
from PIL import Image
import os
import sys
sys.path.insert(1,f'{os.getcwd()}/FastSAM')
#--
from fastsam import FastSAM, FastSAMPrompt
#--
import utils_casia
from importlib import reload; reload(utils_casia)
from utils_casia import resize_image, get_raw_image, get_device, get_results
#--
device = get_device()
fastsam_s = 'weights/FastSAM-s.pt'
fastsam_x = 'weights/FastSAM-x.pt'

In [ ]:
def get_prompt_process(image_path, model, device=None, retina_masks=True, imgsz=1024, conf=0.5, iou=0.6):
    image = get_raw_image(image_path)
    results = get_results(image, model, device=device, retina_masks=retina_masks,
                          imgsz=imgsz, conf=conf, iou=iou)
    prompt_process = FastSAMPrompt(image, results, device=device)
    return prompt_process

def get_prompt_result(prompt_process, text):
    prompt_results = prompt_process.text_prompt(text=text)
    return prompt_results


## Success with the RoboFow Dog Pic

In [ ]:
dog_prompter_fs = get_prompt_process('images/roboflow-dog.jpeg', fastsam_s, device=device)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs = axs.flatten()  # Flatten in case of multi-row subplots
plt.sca(axs[0])  # Set the current axes to the subplot
plt.imshow(get_prompt_result(dog_prompter_fs,'cap')[0])
plt.sca(axs[1])  # Set the current axes to the subplot
plt.imshow(get_prompt_result(dog_prompter_fs,'building')[0]);

<!-- saved-output: cell 61 -->
![CASIA FastSAM-s prompts for cap and building](outputs/roboflow-dog-casia-fastsam-s-prompts.png)



Bigger model does slightly better.

In [ ]:
dog_prompter_fx = get_prompt_process('images/roboflow-dog.jpeg', fastsam_x, device=device)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs = axs.flatten()  # Flatten in case of multi-row subplots
plt.sca(axs[0])  # Set the current axes to the subplot
plt.imshow(get_prompt_result(dog_prompter_fx,'cap')[0])
plt.sca(axs[1])  # Set the current axes to the subplot
plt.imshow(get_prompt_result(dog_prompter_fx,'building')[0]);

<!-- saved-output: cell 63 -->
![CASIA FastSAM-x prompts for cap and building](outputs/roboflow-dog-casia-fastsam-x-prompts.png)



### Success with Random Internet Kitty

In [ ]:
cat_prompter_fx = get_prompt_process('images/kitty.png', fastsam_x, device=device)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs = axs.flatten() 
plt.sca(axs[0])  
plt.imshow(get_prompt_result(cat_prompter_fx,'cat')[0])
axs[0].set_title("Cat")
plt.sca(axs[1]) 
plt.imshow(get_prompt_result(cat_prompter_fx,'bush')[0])
axs[1].set_title("Bush");

<!-- saved-output: cell 65 -->
![CASIA FastSAM-x prompts for cat and bush](outputs/kitty-casia-fastsam-x-prompts.png)



### Failure on Selfie!!!

In [ ]:
slf_prompter_fx = get_prompt_process('images/kevin.jpg', fastsam_x, device=device)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs = axs.flatten() 
plt.sca(axs[0])  
plt.imshow(get_prompt_result(slf_prompter_fx,'face')[0])
axs[0].set_title("Face")
plt.sca(axs[1]) 
plt.imshow(get_prompt_result(slf_prompter_fx,'button')[0])
axs[1].set_title("Button");

<!-- saved-output: cell 67 -->
![CASIA FastSAM-x prompts for face and button](outputs/selfie-casia-fastsam-x-prompts.png)

